# 05. Evaluate all four models

Produces `results/eval.json`, the numbers used in Chapter 5 of the project report:

* precision, recall, mAP50 and mAP50-95 on each validation set (box, and mask for segmentation models)
* precision-recall curves
* confusion counts at confidence 0.25 (the operating threshold)
* inference time per image on CPU and on the GPU backend (40 images, 3 warm-up runs, batch size 1)
* epochs run, best epoch by Ultralytics fitness, and training time

Run it when nothing else is using the GPU, otherwise the latency numbers are distorted.

In [ ]:
from pathlib import Path

# Works whether the notebook is opened from ai_model/ or ai_model/notebooks/.
ROOT = Path.cwd().resolve()
while ROOT.name != "ai_model" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert ROOT.name == "ai_model", "Open this notebook from inside the ai_model folder"

DATA = ROOT / "data"          # downloaded and converted datasets (git-ignored)
RAW = DATA / "raw"            # original downloads
RUNS = ROOT / "runs"          # Ultralytics training runs (git-ignored)
WEIGHTS = ROOT / "weights"    # final model files used by the AI service
RESULTS = ROOT / "results"    # metrics used in the project report
for p in (DATA, RAW, RUNS, WEIGHTS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print("ai_model root:", ROOT)

In [ ]:
import torch

# Apple Silicon uses the MPS backend; an NVIDIA GPU uses CUDA; otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("training device:", DEVICE)

In [ ]:
import glob, json, time
import numpy as np
import pandas as pd
from ultralytics import YOLO

MODELS = {
    "pothole_road_damage": ("segment", DATA / "pothole_road_damage.yaml", ROOT / "Pothole_Segmentation_YOLOv8.v1i.yolov8" / "valid" / "images"),
    "garbage_litter": ("segment", DATA / "garbage_litter" / "data.yaml", DATA / "garbage_litter" / "images" / "val"),
    "open_manhole": ("detect", DATA / "open_manhole" / "data.yaml", DATA / "open_manhole" / "images" / "val"),
    "graffiti": ("detect", DATA / "graffiti" / "data.yaml", DATA / "graffiti" / "images" / "val"),
}

In [ ]:
def evaluate(name, task, data_yaml, val_dir):
    weights = WEIGHTS / f"{name}_model.pt"
    rec = {"task": task}

    # 1. headline metrics at Ultralytics' default validation settings
    r = YOLO(str(weights)).val(data=str(data_yaml), imgsz=640, batch=16, device=DEVICE, plots=True, verbose=False,
                               project=str(RUNS / "val"), name=name, exist_ok=True)
    rec["box"] = dict(p=float(r.box.mp), r=float(r.box.mr), map50=float(r.box.map50), map5095=float(r.box.map))
    if task == "segment":
        rec["mask"] = dict(p=float(r.seg.mp), r=float(r.seg.mr), map50=float(r.seg.map50), map5095=float(r.seg.map))

    # 2. precision-recall curves (every 5th point is enough for plotting)
    rec["curves"] = {key: {"x": np.asarray(x).tolist()[::5], "y": np.asarray(y)[0].tolist()[::5]}
                     for (x, y, _, _), key in zip(r.curves_results, r.curves) if "Precision-Recall" in key}

    # 3. confusion counts at the operating confidence of 0.25
    r25 = YOLO(str(weights)).val(data=str(data_yaml), imgsz=640, batch=16, device=DEVICE, conf=0.25, plots=True,
                                 verbose=False, project=str(RUNS / "val_conf25"), name=name, exist_ok=True)
    rec["confusion"] = np.asarray(r25.confusion_matrix.matrix).tolist()   # [[TP, FP], [FN, -]]
    rec["confusion_conf"] = 0.25

    # 4. latency on CPU and on the GPU backend
    images = sorted(glob.glob(str(val_dir / "*.jpg")))[:40]
    rec["latency"] = {}
    for dev in ("cpu", DEVICE if DEVICE != "cpu" else None):
        if dev is None:
            continue
        m = YOLO(str(weights))
        for im in images[:3]:
            m.predict(im, imgsz=640, device=dev, verbose=False)
        t = []
        for im in images:
            t0 = time.perf_counter()
            m.predict(im, imgsz=640, device=dev, verbose=False)
            t.append((time.perf_counter() - t0) * 1000)
        rec["latency"]["cpu" if dev == "cpu" else "gpu"] = {"mean_ms": float(np.mean(t)), "p95_ms": float(np.percentile(t, 95))}

    # 5. training history
    df = pd.read_csv(RUNS / name / "results.csv")
    df.columns = df.columns.str.strip()
    fit = 0.1 * df["metrics/mAP50(B)"] + 0.9 * df["metrics/mAP50-95(B)"]
    if task == "segment":
        fit = fit + 0.1 * df["metrics/mAP50(M)"] + 0.9 * df["metrics/mAP50-95(M)"]
    rec["epochs_run"] = int(df["epoch"].max())
    rec["best_epoch"] = int(df.loc[fit.idxmax(), "epoch"])
    rec["train_minutes"] = float(df["time"].max() / 60)
    return rec

In [ ]:
results = {}
for name, (task, data_yaml, val_dir) in MODELS.items():
    results[name] = evaluate(name, task, data_yaml, val_dir)
    head = results[name].get("mask", results[name]["box"])
    print(f"{name:22s} mAP50 {head['map50']:.3f}  P {head['p']:.3f}  R {head['r']:.3f}  latency {results[name]['latency']}")
json.dump(results, open(RESULTS / "eval.json", "w"), indent=1)

In [ ]:
rows = []
for name, v in results.items():
    part = "mask" if "mask" in v else "box"
    rows.append({"model": name, "metric": part, **{k: round(x, 3) for k, x in v[part].items()},
                 "cpu_ms": round(v["latency"]["cpu"]["mean_ms"], 1), "epochs": v["epochs_run"], "best_epoch": v["best_epoch"]})
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / "summary.csv", index=False)
summary